In [ ]:
import os


project_dir = '/blue/shenhaowang/qingqisong/be-and-active-travel'
os.chdir(project_dir)

In [ ]:
"""
FIX: Patch torch._pytree compatibility issue
Must run this BEFORE importing torchvision or open_clip

"""

import torch
import torch.utils._pytree

# Add the missing function if it doesn't exist
if not hasattr(torch.utils._pytree, 'register_pytree_node'):
    print(f"  Patching torch.utils._pytree (torch version: {torch.__version__})")

    def _dummy_register_pytree_node(*args, **kwargs):
        """No-op placeholder for older torch versions."""
        pass

    torch.utils._pytree.register_pytree_node = _dummy_register_pytree_node
    print("  ✓ Patch applied successfully")
else:
    print(f"  ✓ No patch needed (torch version: {torch.__version__})")

# Verify the patch works by importing torchvision
try:
    import torchvision
    print(f"  ✓ torchvision imported OK (version: {torchvision.__version__})")
except Exception as e:
    print(f"  ✗ torchvision import failed: {e}")

# Check if open_clip works now
try:
    import open_clip
    print(f"  ✓ open_clip imported OK")
    CLIP_AVAILABLE = True
except ImportError:
    print(f"  ✗ open_clip not installed, will use ResNet50 only")
    CLIP_AVAILABLE = False
except Exception as e:
    print(f"  ✗ open_clip import error: {e}")
    CLIP_AVAILABLE = False

print(f"\n  CUDA available: {torch.cuda.is_available()}")

  Patching torch.utils._pytree (torch version: 2.1.2+cu121)
  ✓ Patch applied successfully
  ✓ torchvision imported OK (version: 0.16.2+cu121)


/home/qingqisong/.local/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


  ✓ open_clip imported OK

  CUDA available: True


In [ ]:

#Complete pipeline:
#1. Load multiple foundation model embeddings
#2. Post-Lasso analysis for each model
#3. Multi-method interpretability analysis
#4. Cross-model comparison


# ============================================================================
# IMPORTS
# ============================================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score, accuracy_score

# Statsmodels
import statsmodels.api as sm

# Scipy
from scipy import stats
from scipy.stats import pearsonr, ttest_ind

# Plotting (optional)
import matplotlib.pyplot as plt
import seaborn as sns

# Check for optional dependencies
try:
    import open_clip
    has_open_clip = True
except:
    has_open_clip = False
    print("Warning: open_clip not available")

try:
    from transformers import AutoModel
    has_transformers = True
except:
    has_transformers = False
    print("Warning: transformers not available")

try:
    import timm
    has_timm = True
except:
    has_timm = False
    print("Warning: timm not available")

print("="*80)
print("  FOUNDATION MODELS: POST-LASSO + INTERPRETABILITY ANALYSIS")
print("="*80)
print(f"\n  Available libraries:")
print(f"    PyTorch:      {torch.__version__}")
print(f"    Open-CLIP:    {has_open_clip}")
print(f"    Transformers: {has_transformers}")
print(f"    TIMM:         {has_timm}")
print(f"    CUDA:         {torch.cuda.is_available()}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\n  Using device: {device}")

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    # Data paths
    DATA_DIR = Path('./')
    TRIP_FILE = DATA_DIR / 'chicago_trip_level_with_embeddings.csv'
    EMB_INDEX_FILE = DATA_DIR / 'embedding_point_index.csv'
    IMAGE_DIR = DATA_DIR / 'satellite_images_points'

    # Existing embeddings
    RESNET_FILE = DATA_DIR / 'raw_resnet_embeddings.npy'
    CLIP_FILE = DATA_DIR / 'raw_clip_embeddings.npy'

    # Models to test
    FOUNDATION_MODELS = [
        'resnet50',
        'clip_vitb32',
        'dinov2_small',
        'dinov2_base',
        'alphaearth'
    ]

    # Analysis parameters
    PCA_COMPONENTS = 100
    LASSO_C = 0.02
    TEST_SIZE = 0.2
    RANDOM_STATE = 42

    # Output directory
    OUTPUT_DIR = DATA_DIR / 'foundation_analysis_output'
    OUTPUT_DIR.mkdir(exist_ok=True)

config = Config()

# ============================================================================
# HELPER: Significance stars
# ============================================================================

def sig_stars(p):
    """Convert p-value to significance stars."""
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    elif p < 0.1:
        return '.'
    return ''

  FOUNDATION MODELS: POST-LASSO + INTERPRETABILITY ANALYSIS

  Available libraries:
    PyTorch:      2.1.2+cu121
    Open-CLIP:    True
    Transformers: True
    TIMM:         True
    CUDA:         True

  Using device: cuda


In [ ]:
# ============================================================================
# DATA LOADING MODULE
# ============================================================================

print(f"\n{'='*80}")
print("STEP 1: LOADING DATA")
print(f"{'='*80}")

# ============================================================================
# Load trip data
# ============================================================================

print(f"\n[1.1] Loading trip-level data...")
trip_df = pd.read_csv(config.TRIP_FILE)
print(f"  Loaded {len(trip_df):,} trips")

# ============================================================================
# Load embedding index
# ============================================================================

print(f"\n[1.2] Loading embedding point index...")
emb_index = pd.read_csv(config.EMB_INDEX_FILE)
print(f"  Loaded {len(emb_index):,} unique locations")

# ============================================================================
# Define feature sets
# ============================================================================

print(f"\n[1.3] Defining feature sets...")

# Person/Household features
person_hh = [c for c in [
    'age', 'Female', 'License', 'Employed', 'Student',
    'White', 'Black', 'Asian', 'Hispanic', 'Bachelor_Above',
    'hhveh', 'Zero_Vehicle_HH', 'hhsize', 'Low_Income', 'High_Income',
    'Home_Owner'
] if c in trip_df.columns]

print(f"    Person/HH features: {len(person_hh)}")

# Trip features
trip_f = [c for c in ['od_dist_mi'] if c in trip_df.columns]
print(f"    Trip features: {len(trip_f)}")

# Built environment features
orig_be = [c for c in trip_df.columns
           if c.startswith('O_') and '_pc' not in c and trip_df[c].notna().mean() > 0.5]
dest_be = [c for c in trip_df.columns
           if c.startswith('D_') and '_pc' not in c and trip_df[c].notna().mean() > 0.5]

print(f"    Origin BE features: {len(orig_be)}")
print(f"    Destination BE features: {len(dest_be)}")

# All tabular features
tabular_feats = person_hh + trip_f + orig_be + dest_be
print(f"    Total tabular features: {len(tabular_feats)}")

# ============================================================================
# Clean data
# ============================================================================

print(f"\n[1.4] Cleaning data...")

# Check for required columns
required_cols = tabular_feats + ['is_transit', 'orig_lat', 'orig_lon', 'dest_lat', 'dest_lon']
missing_cols = [c for c in required_cols if c not in trip_df.columns]

if missing_cols:
    print(f"  WARNING: Missing columns: {missing_cols}")
    required_cols = [c for c in required_cols if c in trip_df.columns]

# Drop rows with missing values
df = trip_df.dropna(subset=required_cols).copy()

# Filter non-negative values for demographic features
for col in person_hh + trip_f:
    if col in df.columns:
        df = df[df[col] >= 0]

print(f"  Clean dataset: {len(df):,} trips")
print(f"  Transit rate: {df['is_transit'].mean()*100:.1f}%")

# ============================================================================
# Create coordinate mapping
# ============================================================================

print(f"\n[1.5] Creating coordinate-to-index mapping...")

coord_to_idx = {}
for idx, row in emb_index.iterrows():
    key = (round(row['lat'], 6), round(row['lon'], 6))
    coord_to_idx[key] = idx

print(f"  Created mapping for {len(coord_to_idx):,} locations")

# ============================================================================
# Map trips to embeddings
# ============================================================================

print(f"\n[1.6] Mapping trips to embedding indices...")

df['orig_emb_idx'] = df.apply(
    lambda r: coord_to_idx.get((round(r['orig_lat'], 6), round(r['orig_lon'], 6)), -1),
    axis=1
)
df['dest_emb_idx'] = df.apply(
    lambda r: coord_to_idx.get((round(r['dest_lat'], 6), round(r['dest_lon'], 6)), -1),
    axis=1
)

# Keep only trips with valid embeddings
valid_mask = (df['orig_emb_idx'] >= 0) & (df['dest_emb_idx'] >= 0)
df = df[valid_mask].copy()

print(f"  Trips with valid embeddings: {len(df):,}")

# Extract indices
orig_idx = df['orig_emb_idx'].values.astype(int)
dest_idx = df['dest_emb_idx'].values.astype(int)

# ============================================================================
# Prepare target and train/test split
# ============================================================================

print(f"\n[1.7] Preparing train/test split...")

y_all = df['is_transit'].values
X_tab_raw = df[tabular_feats].values

train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size=config.TEST_SIZE,
    stratify=y_all,
    random_state=config.RANDOM_STATE
)

y_train = y_all[train_idx]
y_test = y_all[test_idx]

print(f"  Train samples: {len(train_idx):,} (transit: {y_train.mean()*100:.1f}%)")
print(f"  Test samples:  {len(test_idx):,} (transit: {y_test.mean()*100:.1f}%)")

# Scale tabular features
scaler_tab = StandardScaler()
X_tab_train = scaler_tab.fit_transform(X_tab_raw[train_idx])
X_tab_test = scaler_tab.transform(X_tab_raw[test_idx])

print(f"\n  ✓ Data loading complete")


STEP 1: LOADING DATA

[1.1] Loading trip-level data...
  Loaded 20,815 trips

[1.2] Loading embedding point index...
  Loaded 1,147 unique locations

[1.3] Defining feature sets...
    Person/HH features: 16
    Trip features: 1
    Origin BE features: 10
    Destination BE features: 10
    Total tabular features: 37

[1.4] Cleaning data...
  Clean dataset: 13,015 trips
  Transit rate: 17.9%

[1.5] Creating coordinate-to-index mapping...
  Created mapping for 1,147 locations

[1.6] Mapping trips to embedding indices...
  Trips with valid embeddings: 13,013

[1.7] Preparing train/test split...
  Train samples: 10,410 (transit: 17.9%)
  Test samples:  2,603 (transit: 17.9%)

  ✓ Data loading complete


In [ ]:
class SatelliteDataset(Dataset):
    """Dataset for loading satellite images."""

    def __init__(self, index_df, transform=None, image_dir=None):
        self.index_df = index_df.reset_index(drop=True)
        self.transform = transform
        self.image_dir = image_dir or config.IMAGE_DIR

        if self.transform is None:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                )
            ])

    def __len__(self):
        return len(self.index_df)

    def __getitem__(self, idx):
        row = self.index_df.iloc[idx]
        point_id = int(row['point_id'])
        lat = row['lat']
        lon = row['lon']


        img_path = Path(self.image_dir) / f"satellite_point{point_id}_{lat}_{lon}.png"

        if not img_path.exists():
            # ⚠️
            print(f"Warning: Image not found: {img_path}")
            img = torch.zeros(3, 224, 224)
        else:
            from PIL import Image
            img = Image.open(img_path).convert('RGB')
            img = self.transform(img)

        return {
            'image': img,
            'lat': torch.tensor(lat, dtype=torch.float32),
            'lon': torch.tensor(lon, dtype=torch.float32),
            'point_id': point_id
        }
# ============================================================================
# SATELLITE EMBEDDING EXTRACTOR (FIXED)
# ============================================================================

class SatelliteEmbeddingExtractor:
    """Extract embeddings from satellite images using various foundation models."""

    def __init__(self, model_name='resnet50', device='cuda'):
        self.model_name = model_name
        self.device = device
        self.model = None
        self.transform = None
        self.embed_dim = None
        self._is_clip = False
        self._is_timm = False  # NEW: flag for timm models

        self._load()

    def _load(self):
        """Load the specified model."""
        name = self.model_name
        print(f"\n  Loading [{name}] ...")

        if name == 'resnet50':
            base = models.resnet50(weights='IMAGENET1K_V2')
            self.model = nn.Sequential(*list(base.children())[:-1])
            self.embed_dim = 2048
            print("    ✓ ResNet-50 (2048-d)")

        elif name == 'clip_vitb32':
            if has_open_clip:
                try:
                    import open_clip
                    m, _, prep = open_clip.create_model_and_transforms(
                        'ViT-B-32', pretrained='openai'
                    )
                    m = m.to(self.device).eval()
                    self.model = m
                    self.transform = prep
                    self.embed_dim = 512
                    self._is_clip = True
                    print("    ✓ CLIP ViT-B/32 (512-d)")
                except Exception as e:
                    print(f"    ✗ CLIP failed: {e}")

        elif name == 'dinov2_small':
            try:
                self.model = torch.hub.load(
                    'facebookresearch/dinov2', 'dinov2_vits14', verbose=False
                )
                self.embed_dim = 384
                print("    ✓ DINOv2-Small (384-d)")
            except Exception as e:
                print(f"    ✗ DINOv2-Small failed: {e}")

        elif name == 'dinov2_base':
            try:
                self.model = torch.hub.load(
                    'facebookresearch/dinov2', 'dinov2_vitb14', verbose=False
                )
                self.embed_dim = 768
                print("    ✓ DINOv2-Base (768-d)")
            except Exception as e:
                print(f"    ✗ DINOv2-Base failed: {e}")

        elif name == 'alphaearth':
            self._load_alphaearth()

        if self.model is not None and not self._is_clip:
            self.model = self.model.to(self.device).eval()

    def _load_alphaearth(self):
        """Load AlphaEarth model (with fallbacks)."""
        # Try 1: Prithvi-100M (Transformers)
        if has_transformers:
            try:
                from transformers import AutoModel
                m = AutoModel.from_pretrained(
                    "ibm-nasa-geospatial/Prithvi-100M",
                    trust_remote_code=True
                )
                self.model = m.to(self.device).eval()
                self.embed_dim = 768
                self._is_timm = False
                print("    ✓ AlphaEarth → Prithvi-100M (768-d)")
                return
            except Exception as e:
                print(f"    · Prithvi failed: {e}")

        # Try 2: Swin Transformer (TIMM)
        if has_timm:
            try:
                import timm
                m = timm.create_model(
                    'swin_base_patch4_window7_224',
                    pretrained=True,
                    num_classes=0  # Remove classification head
                )
                self.model = m.to(self.device).eval()
                self.embed_dim = 1024
                self._is_timm = True  # Mark as timm model
                print("    ✓ AlphaEarth → Swin-B (1024-d)")
                return
            except Exception as e:
                print(f"    · Swin-B failed: {e}")

        # Try 3: OpenCLIP as fallback
        if has_open_clip:
            try:
                import open_clip
                m, _, prep = open_clip.create_model_and_transforms(
                    'ViT-L-14', pretrained='laion2b_s32b_b82k'
                )
                m = m.to(self.device).eval()
                self.model = m
                self.transform = prep
                self.embed_dim = 768
                self._is_clip = True
                self._is_timm = False
                print("    ✓ AlphaEarth → OpenCLIP ViT-L/14 (768-d)")
                return
            except Exception as e:
                print(f"    · OpenCLIP failed: {e}")

        print("    ✗ All AlphaEarth alternatives failed")

    @torch.no_grad()
    def extract(self, index_df, batch_size=32):
        """Extract embeddings for all locations in index_df."""
        if self.model is None:
            print(f"  ✗ Model not loaded for {self.model_name}")
            return None

        dataset = SatelliteDataset(index_df, transform=self.transform)
        if len(dataset) == 0:
            return None

        loader = DataLoader(
            dataset, batch_size=batch_size,
            shuffle=False, num_workers=2,
            pin_memory=(self.device == 'cuda')
        )

        all_emb, all_lat, all_lon, all_pid = [], [], [], []

        for batch in loader:
            imgs = batch['image'].to(self.device).float()

            # Model-specific forward pass
            if self._is_clip:
                # CLIP models
                emb = self.model.encode_image(imgs)
                emb = emb / emb.norm(dim=-1, keepdim=True)
                emb = emb.float()

            elif 'dinov2' in self.model_name:
                # DINOv2 models
                emb = self.model(imgs)

            elif self._is_timm:
                # TIMM models (e.g., Swin Transformer)
                emb = self.model(imgs)  # Direct call, no pixel_values argument
                if emb.dim() > 2:
                    emb = emb.flatten(1)

            elif self.model_name == 'alphaearth' and has_transformers:
                # Transformers models (e.g., Prithvi)
                try:
                    # Prithvi expects 6 channels
                    imgs6 = imgs.repeat(1, 2, 1, 1)[:, :6]
                    out = self.model(pixel_values=imgs6)
                    emb = out.last_hidden_state[:, 0, :]
                except Exception as e1:
                    try:
                        # Fallback to 3 channels
                        out = self.model(pixel_values=imgs)
                        emb = out.last_hidden_state[:, 0, :]
                    except Exception as e2:
                        # Final fallback: direct call
                        emb = self.model(imgs)
                        if emb.dim() > 2:
                            emb = emb.flatten(1)

            else:
                # Standard CNN models (ResNet, etc.)
                emb = self.model(imgs)
                if emb.dim() > 2:
                    emb = emb.flatten(1)

            all_emb.append(emb.cpu().float().numpy())
            all_lat.extend(batch['lat'].tolist())
            all_lon.extend(batch['lon'].tolist())
            all_pid.extend(batch['point_id'].tolist())

        embeddings = np.vstack(all_emb)

        return {
            'embeddings': embeddings,
            'lats': np.array(all_lat),
            'lons': np.array(all_lon),
            'point_ids': np.array(all_pid),
        }
# ============================================================================
# LOAD OR EXTRACT EMBEDDINGS
# ============================================================================

print(f"\n{'='*80}")
print("STEP 2: LOADING/EXTRACTING FOUNDATION MODEL EMBEDDINGS")
print(f"{'='*80}")

embeddings_dict = {}

# Load existing ResNet embeddings if available
if config.RESNET_FILE.exists():
    print(f"\n[2.1] Loading existing ResNet embeddings...")
    resnet_emb = np.load(config.RESNET_FILE)
    embeddings_dict['resnet50'] = {
        'embeddings': resnet_emb,
        'embed_dim': 2048,
        'lats': emb_index['lat'].values,
        'lons': emb_index['lon'].values,
        'point_ids': emb_index['point_id'].values
    }
    print(f"  ✓ Loaded ResNet: {resnet_emb.shape}")
else:
    print(f"\n[2.1] ResNet embeddings not found, will extract...")

# Load existing CLIP embeddings if available
if config.CLIP_FILE.exists():
    print(f"\n[2.2] Loading existing CLIP embeddings...")
    clip_emb = np.load(config.CLIP_FILE)
    embeddings_dict['clip_vitb32'] = {
        'embeddings': clip_emb,
        'embed_dim': 512,
        'lats': emb_index['lat'].values,
        'lons': emb_index['lon'].values,
        'point_ids': emb_index['point_id'].values
    }
    print(f"  ✓ Loaded CLIP: {clip_emb.shape}")
else:
    print(f"\n[2.2] CLIP embeddings not found, will extract...")

# Extract embeddings for models not yet loaded
for model_name in config.FOUNDATION_MODELS:
    if model_name in embeddings_dict:
        print(f"\n  • {model_name}: already loaded")
        continue

    print(f"\n{'='*80}")
    print(f"Extracting: {model_name.upper()}")
    print(f"{'='*80}")

    try:
        extractor = SatelliteEmbeddingExtractor(
            model_name=model_name,
            device=device
        )

        result = extractor.extract(emb_index, batch_size=32)

        if result is not None:
            embeddings_dict[model_name] = {
                'embeddings': result['embeddings'],
                'embed_dim': extractor.embed_dim,
                'lats': result['lats'],
                'lons': result['lons'],
                'point_ids': result['point_ids']
            }

            # Save to disk
            save_path = config.OUTPUT_DIR / f'embeddings_{model_name}.npy'
            np.save(save_path, result['embeddings'])
            print(f"  ✓ Saved to: {save_path}")

    except Exception as e:
        print(f"  ✗ Failed to extract {model_name}: {e}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*80}")
print(f"Successfully loaded {len(embeddings_dict)} models:")
for name, data in embeddings_dict.items():
    print(f"  • {name:<20} {data['embeddings'].shape}")
print(f"{'='*80}")


STEP 2: LOADING/EXTRACTING FOUNDATION MODEL EMBEDDINGS

[2.1] Loading existing ResNet embeddings...
  ✓ Loaded ResNet: (1147, 2048)

[2.2] Loading existing CLIP embeddings...
  ✓ Loaded CLIP: (1147, 512)

  • resnet50: already loaded

  • clip_vitb32: already loaded

Extracting: DINOV2_SMALL

  Loading [dinov2_small] ...
    ✓ DINOv2-Small (384-d)
  ✓ Saved to: foundation_analysis_output/embeddings_dinov2_small.npy

Extracting: DINOV2_BASE

  Loading [dinov2_base] ...
    ✓ DINOv2-Base (768-d)
  ✓ Saved to: foundation_analysis_output/embeddings_dinov2_base.npy

Extracting: ALPHAEARTH

  Loading [alphaearth] ...
    · Prithvi failed: 'NoneType' object cannot be interpreted as an integer
    ✓ AlphaEarth → Swin-B (1024-d)
  ✓ Saved to: foundation_analysis_output/embeddings_alphaearth.npy

Successfully loaded 5 models:
  • resnet50             (1147, 2048)
  • clip_vitb32          (1147, 512)
  • dinov2_small         (1147, 384)
  • dinov2_base          (1147, 768)
  • alphaearth        

In [ ]:
# ============================================================================
# PCA FEATURE CREATION
# ============================================================================

def create_model_pca_features(embeddings, n_components, train_idx, test_idx,
                               orig_idx, dest_idx):
    """
    Create PCA features from model embeddings for origin and destination.

    Args:
        embeddings: (N_locations, embed_dim) embeddings for all locations
        n_components: number of PCA components
        train_idx: indices of training samples
        test_idx: indices of test samples
        orig_idx: (N_trips,) embedding indices for origins
        dest_idx: (N_trips,) embedding indices for destinations

    Returns:
        X_train_pca: (N_train, n_components*2) training features
        X_test_pca: (N_test, n_components*2) test features
        var_explained: percentage of variance explained
        actual_components: actual number of components used
        pca_object: fitted PCA object
        scaler_object: fitted StandardScaler object
    """
    # Get origin and destination embeddings for train/test
    emb_o_train = embeddings[orig_idx[train_idx]]
    emb_d_train = embeddings[dest_idx[train_idx]]
    emb_o_test = embeddings[orig_idx[test_idx]]
    emb_d_test = embeddings[dest_idx[test_idx]]

    # Stack for PCA fitting (using training data only)
    all_train = np.vstack([emb_o_train, emb_d_train])

    # Scale
    scaler = StandardScaler()
    all_train_scaled = scaler.fit_transform(all_train)

    # Determine max components
    max_comp = min(n_components, all_train_scaled.shape[0], all_train_scaled.shape[1])

    # PCA
    pca = PCA(n_components=max_comp, random_state=config.RANDOM_STATE)
    pca.fit(all_train_scaled)

    var_explained = pca.explained_variance_ratio_.sum() * 100

    # Transform O and D separately
    o_train_pca = pca.transform(scaler.transform(emb_o_train))
    o_test_pca = pca.transform(scaler.transform(emb_o_test))
    d_train_pca = pca.transform(scaler.transform(emb_d_train))
    d_test_pca = pca.transform(scaler.transform(emb_d_test))

    # Concatenate O + D
    X_train_pca = np.column_stack([o_train_pca, d_train_pca])
    X_test_pca = np.column_stack([o_test_pca, d_test_pca])

    return X_train_pca, X_test_pca, var_explained, max_comp, pca, scaler

In [ ]:
# ============================================================================
# POST-LASSO ANALYSIS FOR ALL MODELS
# ============================================================================

print(f"\n{'='*80}")
print("STEP 3: POST-LASSO ANALYSIS")
print(f"{'='*80}")

postlasso_results = {}

for model_name, model_data in embeddings_dict.items():
    print(f"\n{'='*80}")
    print(f"POST-LASSO: {model_name.upper()}")
    print(f"{'='*80}")

    embeddings = model_data['embeddings']
    embed_dim = model_data['embed_dim']

    print(f"  Original embedding dim: {embed_dim}")

    # ========================================================================
    # Create PCA features
    # ========================================================================
    print(f"\n  [1] Creating PCA features...")

    X_tr_img, X_te_img, var_exp, n_comp, pca_obj, scaler_obj = create_model_pca_features(
        embeddings, config.PCA_COMPONENTS, train_idx, test_idx, orig_idx, dest_idx
    )

    print(f"    PCA components: {n_comp}")
    print(f"    Variance explained: {var_exp:.2f}%")
    print(f"    Image feature shape: {X_tr_img.shape}")

    # Combine with tabular features
    X_tr_full = np.column_stack([X_tab_train, X_tr_img])
    X_te_full = np.column_stack([X_tab_test, X_te_img])

    # Scale
    scaler_full = StandardScaler()
    X_tr_scaled = scaler_full.fit_transform(X_tr_full)
    X_te_scaled = scaler_full.transform(X_te_full)

    # Feature names
    img_feat_names = (
        [f'O_{model_name}_pc{i+1}' for i in range(n_comp)] +
        [f'D_{model_name}_pc{i+1}' for i in range(n_comp)]
    )
    all_feat_names = tabular_feats + img_feat_names

    # ========================================================================
    # Step 1: Lasso variable selection
    # ========================================================================
    print(f"\n  [2] Lasso variable selection (C={config.LASSO_C})...")

    lasso = LogisticRegression(
        penalty='l1', C=config.LASSO_C, solver='liblinear',
        max_iter=3000, random_state=config.RANDOM_STATE
    )
    lasso.fit(X_tr_scaled, y_train)

    coefs = lasso.coef_[0]
    selected_mask = coefs != 0
    n_selected = selected_mask.sum()

    # Count by category
    n_tab_selected = sum(1 for i, name in enumerate(all_feat_names)
                         if selected_mask[i] and name in tabular_feats)
    n_img_selected = sum(1 for i, name in enumerate(all_feat_names)
                         if selected_mask[i] and model_name in name)

    print(f"    Variables selected: {n_selected}")
    print(f"      Tabular: {n_tab_selected}")
    print(f"      Image: {n_img_selected}")

    if n_selected == 0:
        print(f"    ⚠️ No variables selected, skipping this model")
        continue

    # ========================================================================
    # Step 2: Post-Lasso estimation
    # ========================================================================
    print(f"\n  [3] Post-Lasso estimation with statsmodels...")

    selected_indices = np.where(selected_mask)[0]
    selected_names = [all_feat_names[i] for i in selected_indices]

    X_tr_selected = X_tr_scaled[:, selected_indices]
    X_te_selected = X_te_scaled[:, selected_indices]

    X_tr_sm = sm.add_constant(X_tr_selected, has_constant='add')
    X_te_sm = sm.add_constant(X_te_selected, has_constant='add')

    try:
        logit = sm.Logit(y_train, X_tr_sm)
        result = logit.fit(disp=0, maxiter=1000, method='lbfgs')

        # ====================================================================
        # Step 3: Coefficient analysis
        # ====================================================================
        print(f"\n  [4] Analyzing coefficients...")

        coef_data = []
        for i, name in enumerate(selected_names):
            idx = i + 1  # +1 for intercept

            # Categorize
            if name in tabular_feats:
                if name in person_hh:
                    cat = 'Person/HH'
                elif name in trip_f:
                    cat = 'Trip'
                elif name in orig_be or name in dest_be:
                    cat = 'Zone BE'
                else:
                    cat = 'Tabular'
            else:
                cat = model_name

            coef_data.append({
                'variable': name,
                'coef': result.params[idx],
                'std_err': result.bse[idx],
                'z': result.tvalues[idx],
                'p_value': result.pvalues[idx],
                'category': cat
            })

        coef_df = pd.DataFrame(coef_data)
        coef_df['abs_coef'] = np.abs(coef_df['coef'])
        coef_df['sig'] = coef_df['p_value'].apply(sig_stars)

        # Count significant image features
        img_df = coef_df[coef_df['category'] == model_name]
        sig_img = img_df[img_df['p_value'] < 0.05]

        print(f"    Image features significant (p<0.05): {len(sig_img)} / {len(img_df)}")

        # ====================================================================
        # Step 4: Performance metrics
        # ====================================================================
        print(f"\n  [5] Computing performance metrics...")

        # Train metrics
        ll_train = result.llf
        ll_null_train = result.llnull
        pseudo_r2_train = result.prsquared
        aic = result.aic
        bic = result.bic

        # Test predictions
        y_prob_test = result.predict(X_te_sm)
        auc_test = roc_auc_score(y_test, y_prob_test)
        acc_test = accuracy_score(y_test, (y_prob_test > 0.5).astype(int))

        # Test LL
        eps = 1e-15
        y_prob_clipped = np.clip(y_prob_test, eps, 1 - eps)
        ll_test = np.sum(y_test * np.log(y_prob_clipped) +
                         (1 - y_test) * np.log(1 - y_prob_clipped))

        # Test Pseudo R²
        p_null = y_train.mean()
        ll_null_test = np.sum(y_test * np.log(p_null) +
                              (1 - y_test) * np.log(1 - p_null))
        pseudo_r2_test = 1 - (ll_test / ll_null_test)

        print(f"    LL (train): {ll_train:.2f}")
        print(f"    LL (test):  {ll_test:.2f}")
        print(f"    Pseudo R² (train): {pseudo_r2_train:.4f}")
        print(f"    Pseudo R² (test):  {pseudo_r2_test:.4f}")
        print(f"    AUC (test): {auc_test:.4f}")

        # ====================================================================
        # Store results
        # ====================================================================
        postlasso_results[model_name] = {
            'model_name': model_name,
            'embed_dim': embed_dim,
            'pca_components': n_comp,
            'variance_explained': var_exp,
            'n_selected': n_selected,
            'n_tab_selected': n_tab_selected,
            'n_img_selected': n_img_selected,
            'n_img_significant': len(sig_img),
            'll_train': ll_train,
            'll_test': ll_test,
            'll_null_train': ll_null_train,
            'll_null_test': ll_null_test,
            'pseudo_r2_train': pseudo_r2_train,
            'pseudo_r2_test': pseudo_r2_test,
            'aic': aic,
            'bic': bic,
            'auc_test': auc_test,
            'acc_test': acc_test,
            'coef_df': coef_df,
            'sig_img_features': sig_img,
            'pca_object': pca_obj,
            'scaler_object': scaler_obj,
            'selected_names': selected_names,
            'statsmodels_result': result
        }

        print(f"    ✓ Analysis complete for {model_name}")

    except Exception as e:
        print(f"    ✗ Error in statsmodels fitting: {e}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*80}")
print(f"POST-LASSO COMPLETE: {len(postlasso_results)} models analyzed")
print(f"{'='*80}")


STEP 3: POST-LASSO ANALYSIS

POST-LASSO: RESNET50
  Original embedding dim: 2048

  [1] Creating PCA features...
    PCA components: 100
    Variance explained: 90.71%
    Image feature shape: (10410, 200)

  [2] Lasso variable selection (C=0.02)...
    Variables selected: 61
      Tabular: 20
      Image: 41

  [3] Post-Lasso estimation with statsmodels...

  [4] Analyzing coefficients...
    Image features significant (p<0.05): 20 / 41

  [5] Computing performance metrics...
    LL (train): -3370.54
    LL (test):  -891.79
    Pseudo R² (train): 0.3113
    Pseudo R² (test):  0.2709
    AUC (test): 0.8602
    ✓ Analysis complete for resnet50

POST-LASSO: CLIP_VITB32
  Original embedding dim: 512

  [1] Creating PCA features...
    PCA components: 100
    Variance explained: 95.76%
    Image feature shape: (10410, 200)

  [2] Lasso variable selection (C=0.02)...
    Variables selected: 63
      Tabular: 19
      Image: 44

  [3] Post-Lasso estimation with statsmodels...

  [4] Analyzi

In [ ]:
# ============================================================================
# CROSS-MODEL COMPARISON TABLE
# ============================================================================

print(f"\n{'='*80}")
print("STEP 4: CROSS-MODEL COMPARISON")
print(f"{'='*80}")

if len(postlasso_results) == 0:
    print("  ⚠️ No models successfully analyzed")
else:
    # Create comparison dataframe
    comparison_data = []

    for model_name, res in postlasso_results.items():
        comparison_data.append({
            'Model': model_name,
            'Embed Dim': res['embed_dim'],
            'PCA Comps': res['pca_components'],
            'Var Exp %': f"{res['variance_explained']:.1f}",
            'Total Sel': res['n_selected'],
            'Tab Sel': res['n_tab_selected'],
            'Img Sel': res['n_img_selected'],
            'Img Sig': res['n_img_significant'],
            'ρ² (train)': f"{res['pseudo_r2_train']:.4f}",
            'ρ² (test)': f"{res['pseudo_r2_test']:.4f}",
            'AUC (test)': f"{res['auc_test']:.4f}",
            'AIC': f"{res['aic']:.1f}",
            'BIC': f"{res['bic']:.1f}"
        })

    comp_df = pd.DataFrame(comparison_data)

    # Sort by test Pseudo R²
    comp_df['ρ²_test_numeric'] = comp_df['ρ² (test)'].astype(float)
    comp_df = comp_df.sort_values('ρ²_test_numeric', ascending=False)
    comp_df = comp_df.drop(columns=['ρ²_test_numeric'])

    print("\n" + comp_df.to_string(index=False))

    # Save to CSV
    output_csv = config.OUTPUT_DIR / 'postlasso_model_comparison.csv'
    comp_df.to_csv(output_csv, index=False)
    print(f"\n  ✓ Saved comparison table to: {output_csv}")

    # Best model
    best_model = comp_df.iloc[0]['Model']
    best_r2 = comp_df.iloc[0]['ρ² (test)']

    print(f"\n  🏆 Best model (by test ρ²): {best_model} ({best_r2})")


STEP 4: CROSS-MODEL COMPARISON

       Model  Embed Dim  PCA Comps Var Exp %  Total Sel  Tab Sel  Img Sel  Img Sig ρ² (train) ρ² (test) AUC (test)    AIC    BIC
 dinov2_base        768        100      93.3         69       20       49       16     0.3132    0.2758     0.8623 6862.4 7369.9
  alphaearth       1024        100      92.5         73       20       53       22     0.3149    0.2738     0.8615 6853.1 7389.7
dinov2_small        384        100      94.9         68       20       48       25     0.3160    0.2711     0.8585 6833.1 7333.3
    resnet50       2048        100      90.7         61       20       41       20     0.3113    0.2709     0.8602 6865.1 7314.6
 clip_vitb32        512        100      95.8         63       19       44       22     0.3148    0.2650     0.8548 6834.5 7298.5

  ✓ Saved comparison table to: foundation_analysis_output/postlasso_model_comparison.csv

  🏆 Best model (by test ρ²): dinov2_base (0.2758)


In [ ]:
# ============================================================================
# CLIP ZERO-SHOT SEMANTIC ALIGNMENT
# ============================================================================

print(f"\n{'='*80}")
print("STEP 6: CLIP ZERO-SHOT SEMANTIC ALIGNMENT")
print(f"{'='*80}")

if 'clip_vitb32' in embeddings_dict and has_open_clip:
    try:
        import open_clip

        # Load CLIP model
        print(f"\n[6.1] Loading CLIP model for text encoding...")
        model, _, preprocess = open_clip.create_model_and_transforms(
            'ViT-B-32', pretrained='openai'
        )
        model = model.to(device).eval()
        # open_clip.tokenize
        tokenizer = open_clip.tokenize

        # Define urban concept texts
        urban_concepts_text = [
            "dense urban downtown with tall buildings",
            "suburban residential area with single-family homes",
            "mixed-use neighborhood with shops and apartments",
            "highway interchange with parking lots",
            "tree-lined streets with sidewalks",
            "grid street network with frequent intersections",
            "cul-de-sac suburban development",
            "transit-oriented development with bus stops",
            "walkable urban core",
            "car-dependent sprawl",
            "bike lanes visible on streets",
            "parking lot dominates the view",
            "bus stop or transit station visible",
            "green parks and trees",
            "pedestrian-friendly neighborhood"
        ]

        print(f"  Defined {len(urban_concepts_text)} urban concepts")

        # Encode texts
        print(f"\n[6.2] Encoding text concepts...")
        text_tokens = tokenizer(urban_concepts_text).to(device)
        with torch.no_grad():
            text_features = model.encode_text(text_tokens)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        # Get CLIP embeddings
        clip_embeddings = embeddings_dict['clip_vitb32']['embeddings']
        clip_embeddings_tensor = torch.from_numpy(clip_embeddings).float().to(device)
        clip_embeddings_tensor = clip_embeddings_tensor / clip_embeddings_tensor.norm(
            dim=-1, keepdim=True
        )

        # Compute similarities
        print(f"\n[6.3] Computing text-image similarities...")
        similarities = clip_embeddings_tensor @ text_features.T
        similarities_np = similarities.cpu().numpy()

        # ====================================================================
        # Analyze concept-mode choice correlations
        # ====================================================================
        print(f"\n[6.4] Analyzing concept-mode choice correlations...")

        concept_mode_correlations = []

        for i, concept_text in enumerate(urban_concepts_text):
            # Map to trips (using origins)
            trip_similarities = []
            trip_modes = []

            for trip_idx in range(len(df)):
                orig_emb_idx = df.iloc[trip_idx]['orig_emb_idx']
                if 0 <= orig_emb_idx < len(similarities_np):
                    trip_similarities.append(similarities_np[orig_emb_idx, i])
                    trip_modes.append(df.iloc[trip_idx]['is_transit'])

            if len(trip_similarities) < 100:
                continue

            # Compute correlation
            corr, p_value = pearsonr(trip_similarities, trip_modes)

            concept_mode_correlations.append({
                'concept': concept_text,
                'correlation': corr,
                'p_value': p_value,
                'sig': sig_stars(p_value)
            })

        # Sort by correlation
        concept_df = pd.DataFrame(concept_mode_correlations).sort_values(
            'correlation', ascending=False
        )

        # Print results
        print(f"\n  {'Concept':<55} {'Corr':>10} {'p-value':>12} {'Sig':>5}")
        print(f"  {'-'*85}")

        for _, row in concept_df.iterrows():
            print(f"  {row['concept']:<55} {row['correlation']:>+10.3f} "
                  f"{row['p_value']:>12.6f} {row['sig']:>5}")

        print(f"\n  TOP TRANSIT-PROMOTING CONCEPTS:")
        for _, row in concept_df.head(5).iterrows():
            print(f"    • {row['concept']}: r = {row['correlation']:+.3f} "
                  f"(p = {row['p_value']:.4f})")

        print(f"\n  TOP AUTO-PROMOTING CONCEPTS:")
        for _, row in concept_df.tail(5).iterrows():
            print(f"    • {row['concept']}: r = {row['correlation']:+.3f} "
                  f"(p = {row['p_value']:.4f})")

        # Save results
        clip_csv = config.OUTPUT_DIR / 'clip_zeroshot_concepts.csv'
        concept_df.to_csv(clip_csv, index=False)
        print(f"\n  ✓ Saved CLIP zero-shot results to: {clip_csv}")

    except Exception as e:
        print(f"\n  ✗ CLIP zero-shot analysis failed: {e}")
        import traceback
        traceback.print_exc()
else:
    print("\n  ⚠️ CLIP not available, skipping zero-shot analysis")


STEP 6: CLIP ZERO-SHOT SEMANTIC ALIGNMENT

[6.1] Loading CLIP model for text encoding...
  Defined 15 urban concepts

[6.2] Encoding text concepts...

[6.3] Computing text-image similarities...

[6.4] Analyzing concept-mode choice correlations...

  Concept                                                       Corr      p-value   Sig
  -------------------------------------------------------------------------------------
  dense urban downtown with tall buildings                    +0.009     0.329079      
  walkable urban core                                         +0.003     0.695265      
  parking lot dominates the view                              +0.001     0.865233      
  transit-oriented development with bus stops                 -0.003     0.730284      
  grid street network with frequent intersections             -0.011     0.201207      
  highway interchange with parking lots                       -0.028     0.001433    **
  bike lanes visible on streets                

In [ ]:
# ============================================================================
# FIXED: Image Path Resolution
# ============================================================================

def find_image_path(point_id, lat, lon, image_dir):
    """
    Find satellite image file with flexible naming patterns.

    Tries multiple patterns:
    1. satellite_point{id}_{lat}_{lon}.png
    2. {id}.png
    3. point_{id}.png
    """
    image_dir = Path(image_dir)
    point_id = int(point_id)

    # Pattern 1: satellite_point{id}_{lat}_{lon}.png
    pattern1 = image_dir / f"satellite_point{point_id}_{lat}_{lon}.png"
    if pattern1.exists():
        return pattern1



    return None


# ============================================================================
# UPDATED: Visual Interpretation with Fixed Image Loading
# ============================================================================

def interpret_pc_by_visualization_fixed(model_name, component_name, n_samples=10):
    """
    Interpret a PCA component by visualizing extreme images.
    FIXED: Proper image path resolution
    """
    print(f"\n{'='*80}")
    print(f"VISUAL INTERPRETATION: {component_name} ({model_name})")
    print(f"{'='*80}")

    # Get Post-Lasso results
    result_dict = postlasso_results[model_name]
    pca_obj = result_dict['pca_object']
    scaler_obj = result_dict['scaler_object']

    # Get embeddings
    embeddings = embeddings_dict[model_name]['embeddings']

    # Extract PC number and location
    if component_name.startswith('O_'):
        is_origin = True
        pc_num = int(component_name.split('_pc')[1]) - 1
    else:
        is_origin = False
        pc_num = int(component_name.split('_pc')[1]) - 1

    # Transform to PCA space
    all_scaled = scaler_obj.transform(embeddings)
    all_pca = pca_obj.transform(all_scaled)
    pc_values = all_pca[:, pc_num]

    print(f"\n  PC{pc_num+1} Statistics:")
    print(f"    Range: [{pc_values.min():.3f}, {pc_values.max():.3f}]")
    print(f"    Mean:  {pc_values.mean():.3f}")
    print(f"    Std:   {pc_values.std():.3f}")

    # Find extreme indices
    sorted_indices = np.argsort(pc_values)
    low_indices = sorted_indices[:n_samples]
    high_indices = sorted_indices[-n_samples:]

    print(f"\n  Analyzing {n_samples} locations with LOWEST values:")
    print(f"    PC values: {pc_values[low_indices]}")

    print(f"\n  Analyzing {n_samples} locations with HIGHEST values:")
    print(f"    PC values: {pc_values[high_indices]}")

    # ========================================================================
    # Built Environment Comparison
    # ========================================================================
    print(f"\n  {'='*70}")
    print(f"  BUILT ENVIRONMENT COMPARISON")
    print(f"  {'='*70}")

    be_vars_to_check = [
        'avg_building_density', 'avg_intersection_density',
        'dist_cbd_mi', 'avg_land_use_diversity', 'dist_park_mi'
    ]

    prefix = 'O_' if is_origin else 'D_'

    print(f"\n  {'BE Variable':<35} {'LOW PC':<15} {'HIGH PC':<15} {'Difference'}")
    print(f"  {'-'*75}")

    be_interpretations = []

    for be_var in be_vars_to_check:
        col = f'{prefix}{be_var}'
        if col in df.columns:
            low_trips = df[df['orig_emb_idx' if is_origin else 'dest_emb_idx'].isin(low_indices)]
            high_trips = df[df['orig_emb_idx' if is_origin else 'dest_emb_idx'].isin(high_indices)]

            if len(low_trips) > 0 and len(high_trips) > 0:
                low_mean = low_trips[col].mean()
                high_mean = high_trips[col].mean()
                diff = high_mean - low_mean

                # Interpret direction
                if abs(diff) > 0.1 * abs(low_mean):
                    direction = "→ HIGH" if diff > 0 else "→ LOW"

                    # Store for interpretation
                    if abs(diff) > 0.2 * abs(low_mean):  # >20% difference
                        dir_text = "increases" if diff > 0 else "decreases"
                        be_interpretations.append(
                            f"{be_var} {dir_text} from LOW to HIGH PC ({diff:+.2f})"
                        )
                else:
                    direction = "≈ same"

                print(f"  {be_var:<35} {low_mean:>14.3f} {high_mean:>14.3f} "
                      f"{diff:>+10.3f} {direction}")

    # ========================================================================
    # Mode Choice Association
    # ========================================================================
    print(f"\n  {'='*70}")
    print(f"  MODE CHOICE ASSOCIATION")
    print(f"  {'='*70}")

    all_trip_pc_values = []
    all_trip_modes = []

    emb_col = 'orig_emb_idx' if is_origin else 'dest_emb_idx'

    for idx, row in df.iterrows():
        emb_idx = row[emb_col]
        if 0 <= emb_idx < len(pc_values):
            all_trip_pc_values.append(pc_values[emb_idx])
            all_trip_modes.append(row['is_transit'])

    all_trip_pc_values = np.array(all_trip_pc_values)
    all_trip_modes = np.array(all_trip_modes)

    low_mask = all_trip_pc_values < np.percentile(all_trip_pc_values, 25)
    high_mask = all_trip_pc_values > np.percentile(all_trip_pc_values, 75)

    transit_rate_low = all_trip_modes[low_mask].mean()
    transit_rate_high = all_trip_modes[high_mask].mean()

    from scipy.stats import ttest_ind
    t_stat, p_val = ttest_ind(all_trip_modes[low_mask], all_trip_modes[high_mask])

    print(f"\n  Transit rate (LOW PC quartile):  {transit_rate_low*100:.1f}%")
    print(f"  Transit rate (HIGH PC quartile): {transit_rate_high*100:.1f}%")
    print(f"  Difference: {(transit_rate_high - transit_rate_low)*100:+.1f} percentage points")
    print(f"  T-test: t={t_stat:.3f}, p={p_val:.6f} {sig_stars(p_val)}")

    # ========================================================================
    # FIXED: Collect Image Paths
    # ========================================================================
    print(f"\n  {'='*70}")
    print(f"  SAMPLE IMAGES")
    print(f"  {'='*70}")

    low_image_paths = []
    high_image_paths = []

    print(f"\n  Searching for LOW PC images...")
    for idx in low_indices[:5]:
        point_id = emb_index.iloc[idx]['point_id']
        lat = emb_index.iloc[idx]['lat']
        lon = emb_index.iloc[idx]['lon']

        # Try to find image with flexible naming
        img_path = find_image_path(point_id, lat, lon, config.IMAGE_DIR)

        if img_path:
            low_image_paths.append((img_path, pc_values[idx], point_id, lat, lon))
            print(f"    ✓ Found: {img_path.name}")
        else:
            print(f"    ✗ Not found: point_id={point_id}, lat={lat:.6f}, lon={lon:.6f}")

    print(f"\n  Searching for HIGH PC images...")
    for idx in high_indices[-5:]:
        point_id = emb_index.iloc[idx]['point_id']
        lat = emb_index.iloc[idx]['lat']
        lon = emb_index.iloc[idx]['lon']

        img_path = find_image_path(point_id, lat, lon, config.IMAGE_DIR)

        if img_path:
            high_image_paths.append((img_path, pc_values[idx], point_id, lat, lon))
            print(f"    ✓ Found: {img_path.name}")
        else:
            print(f"    ✗ Not found: point_id={point_id}, lat={lat:.6f}, lon={lon:.6f}")

    print(f"\n  Summary:")
    print(f"    LOW PC images found:  {len(low_image_paths)}/5")
    print(f"    HIGH PC images found: {len(high_image_paths)}/5")

    # ========================================================================
    # Create Visualization
    # ========================================================================

    if len(low_image_paths) >= 3 and len(high_image_paths) >= 3:
        from PIL import Image
        import matplotlib.pyplot as plt

        n_show = min(5, len(low_image_paths), len(high_image_paths))

        fig, axes = plt.subplots(2, n_show, figsize=(4*n_show, 8))

        fig.suptitle(
            f'{component_name} ({model_name}): Visual Interpretation\n'
            f'Transit: LOW={transit_rate_low*100:.1f}% vs HIGH={transit_rate_high*100:.1f}%',
            fontsize=14, fontweight='bold'
        )

        # Handle single column case
        if n_show == 1:
            axes = axes.reshape(2, 1)

        # Plot LOW PC images
        for i in range(n_show):
            if i < len(low_image_paths):
                img_path, val, pid, lat, lon = low_image_paths[i]
                img = Image.open(img_path)
                axes[0, i].imshow(img)
                axes[0, i].set_title(f'PC={val:.2f}\nID:{pid}', fontsize=9)
                axes[0, i].axis('off')

        axes[0, 0].text(-0.1, 0.5, 'LOW PC\n(Pro-Transit)',
                        transform=axes[0, 0].transAxes,
                        fontsize=12, fontweight='bold', va='center',
                        rotation=90)

        # Plot HIGH PC images
        for i in range(n_show):
            if i < len(high_image_paths):
                img_path, val, pid, lat, lon = high_image_paths[i]
                img = Image.open(img_path)
                axes[1, i].imshow(img)
                axes[1, i].set_title(f'PC={val:.2f}\nID:{pid}', fontsize=9)
                axes[1, i].axis('off')

        axes[1, 0].text(-0.1, 0.5, 'HIGH PC\n(Pro-Auto)',
                        transform=axes[1, 0].transAxes,
                        fontsize=12, fontweight='bold', va='center',
                        rotation=90)

        plt.tight_layout()

        # Save
        output_path = config.OUTPUT_DIR / f'interpretation_{model_name}_{component_name}.png'
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"\n  ✓ Saved visualization to: {output_path}")
        plt.close()
    else:
        print(f"\n  ⚠️ Insufficient images for visualization (need ≥3 per group)")

    # ========================================================================
    # Interpretation Summary
    # ========================================================================
    print(f"\n  {'='*70}")
    print(f"  INTERPRETATION SUMMARY: {component_name}")
    print(f"  {'='*70}")

    interpretation = []

    # BE interpretations
    interpretation.extend(be_interpretations)

    # Mode choice
    if abs(transit_rate_high - transit_rate_low) > 0.03 and p_val < 0.05:
        direction = "promotes" if transit_rate_high > transit_rate_low else "discourages"
        interpretation.append(
            f"Higher PC values {direction} transit use "
            f"({(transit_rate_high - transit_rate_low)*100:+.1f}pp, p={p_val:.4f})"
        )

    if interpretation:
        print(f"\n  This component appears to represent:")
        for i, interp in enumerate(interpretation, 1):
            print(f"    {i}. {interp}")
    else:
        print(f"\n  Weak correlations - no clear interpretation")

    return {
        'component': component_name,
        'model': model_name,
        'pc_num': pc_num,
        'low_images': low_image_paths,
        'high_images': high_image_paths,
        'transit_effect': transit_rate_high - transit_rate_low,
        'transit_pval': p_val,
        'interpretation': interpretation,
        'be_diffs': be_interpretations
    }


# ============================================================================
# DIAGNOSTIC: Check actual image file names
# ============================================================================

print(f"\n{'='*80}")
print("DIAGNOSTIC: Checking Image File Naming Pattern")
print(f"{'='*80}")

image_files = list(config.IMAGE_DIR.glob("*.png"))

if len(image_files) > 0:
    print(f"\n  Found {len(image_files)} image files")
    print(f"\n  First 10 file names:")
    for i, img_path in enumerate(image_files[:10], 1):
        print(f"    {i}. {img_path.name}")

    # Parse first file to understand pattern
    first_name = image_files[0].name
    print(f"\n  Analyzing pattern from: {first_name}")

    if 'satellite_point' in first_name:
        parts = first_name.replace('satellite_point', '').replace('.png', '').split('_')
        print(f"    Pattern detected: satellite_point{{id}}_{{lat}}_{{lon}}.png")
        print(f"    Parts: {parts}")

    # Check if our emb_index point_ids match
    sample_point_id = emb_index.iloc[0]['point_id']
    sample_lat = emb_index.iloc[0]['lat']
    sample_lon = emb_index.iloc[0]['lon']

    print(f"\n  Sample from emb_index:")
    print(f"    point_id: {sample_point_id}")
    print(f"    lat: {sample_lat}")
    print(f"    lon: {sample_lon}")

    # Test our find function
    found = find_image_path(sample_point_id, sample_lat, sample_lon, config.IMAGE_DIR)
    if found:
        print(f"    ✓ Image found: {found.name}")
    else:
        print(f"    ✗ Image NOT found")
        # Try manual search
        pattern = f"*{sample_point_id}*"
        matches = list(config.IMAGE_DIR.glob(pattern))
        print(f"    Manual search '*{sample_point_id}*': {len(matches)} matches")
        if matches:
            print(f"      {matches[0].name}")

else:
    print(f"\n  ✗ No PNG files found in {config.IMAGE_DIR}")
    print(f"  Please check IMAGE_DIR configuration")

# ============================================================================
# RE-RUN with fixed image loading
# ============================================================================

print(f"\n{'='*80}")
print("RE-RUNNING INTERPRETATION WITH FIXED IMAGE LOADING")
print(f"{'='*80}")

interpretation_results_fixed = {}

for model_name in ['resnet50', 'clip_vitb32']:
    if model_name not in postlasso_results:
        continue

    sig_img = postlasso_results[model_name]['sig_img_features']

    if len(sig_img) == 0:
        continue

    sig_sorted = sig_img.sort_values('p_value')

    model_interpretations = []

    for _, row in sig_sorted.head(3).iterrows():
        component_name = row['variable']

        try:
            result = interpret_pc_by_visualization_fixed(
                model_name, component_name, n_samples=10
            )
            model_interpretations.append(result)
        except Exception as e:
            print(f"\n  ✗ Error interpreting {component_name}: {e}")
            import traceback
            traceback.print_exc()

    interpretation_results_fixed[model_name] = model_interpretations

print(f"\n{'='*80}")
print("INTERPRETATION COMPLETE")
print(f"{'='*80}")


DIAGNOSTIC: Checking Image File Naming Pattern

  Found 1147 image files

  First 10 file names:
    1. satellite_point60_41.870554_-87.616596.png
    2. satellite_point107_41.959585_-87.810459.png
    3. satellite_point235_41.963974_-87.691821.png
    4. satellite_point494_41.936086_-87.66611.png
    5. satellite_point912_41.919629_-87.750976.png
    6. satellite_point1097_41.738289_-88.0217.png
    7. satellite_point1004_41.729854_-87.771261.png
    8. satellite_point105_41.963374_-87.670184.png
    9. satellite_point1124_41.911787_-87.714988.png
    10. satellite_point503_41.85599_-87.664906.png

  Analyzing pattern from: satellite_point60_41.870554_-87.616596.png
    Pattern detected: satellite_point{id}_{lat}_{lon}.png
    Parts: ['60', '41.870554', '-87.616596']

  Sample from emb_index:
    point_id: 1000.0
    lat: 41.855161
    lon: -87.732243
    ✓ Image found: satellite_point1000_41.855161_-87.732243.png

RE-RUNNING INTERPRETATION WITH FIXED IMAGE LOADING

VISUAL INTERPRETA